# Train the "Hey FRIDAY" wake word (Google Colab)

Trains a custom [openWakeWord](https://github.com/dscripka/openWakeWord) model for the phrase **"hey friday"** and exports `hey_friday.onnx`. Drop that file into FRIDAY at `models/wakeword/hey_friday.onnx` and point the wake-word seam at it (`FRIDAY_WAKEWORD_MODEL=models/wakeword/hey_friday.onnx`).

**Runtime:** Colab → *Runtime ▸ Change runtime type ▸ GPU* (T4 is plenty). The whole run is ~15-30 min.

**How it works (openWakeWord custom-model recipe):**
1. Synthesize many spoken samples of "hey friday" with a TTS (piper-sample-generator) across varied speakers.
2. Mix in noise/reverb augmentation so the model is robust.
3. Use the shared openWakeWord audio-embedding model to turn clips into features.
4. Train a small classifier (positives = "hey friday", negatives = a large speech/noise corpus).
5. Export to ONNX and test it.

> If an API/URL below drifts, follow the upstream `automatic_model_training.ipynb` in the openWakeWord repo — this notebook mirrors it, specialized to "hey friday".

## 1. Install dependencies

In [ ]:
# openWakeWord — install WITHOUT tflite-runtime (it has no wheel for Colab's
# Python), then add the ONNX runtime we actually use. --no-deps skips the
# tflite-runtime pin; we supply openWakeWord's real runtime deps explicitly.
!pip install -q --no-deps openwakeword==0.6.0
!pip install -q onnx onnxruntime scipy scikit-learn tqdm requests

# piper-sample-generator: synthesizes many varied spoken "hey friday" clips.
!pip install -q piper-phonemize piper-tts 2>/dev/null || echo "piper extras: skipped (optional)"
!git clone -q https://github.com/rhasspy/piper-sample-generator.git || true
# Its requirements.txt can be absent/renamed across revisions — install only if present.
!if [ -f piper-sample-generator/requirements.txt ]; then \
    pip install -q -r piper-sample-generator/requirements.txt; \
  else echo "no requirements.txt — relying on Colab's torch + piper extras"; fi

# The libritts voice the generator speaks the phrase in.
!mkdir -p piper-sample-generator/models out
!wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt \
    https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt || true
import os; os.makedirs('out', exist_ok=True); print('ok')

## 2. Download the openWakeWord base models (shared feature extractor)

In [ ]:
import openwakeword
from openwakeword.utils import download_models
download_models()  # melspectrogram + embedding models the trainer/inference share
print('base models ready')

## 3. Synthesize positive samples of "hey friday"

`piper-sample-generator` produces many varied utterances of the phrase. Bump `--max-samples` for a stronger model (1000-2000 is a good range).

In [ ]:
WAKE_PHRASE = 'hey friday'
N_POSITIVES = 1500
!cd piper-sample-generator && python generate_samples.py \
    "{WAKE_PHRASE}" \
    --model models/en_US-libritts_r-medium.pt \
    --max-samples {N_POSITIVES} \
    --batch-size 64 \
    --output-dir ../out/positives
import glob; print('positive clips:', len(glob.glob('out/positives/*.wav')))

## 4. Get negatives + augmentation audio (background speech, noise, room impulses)

openWakeWord ships precomputed negative *features* and augmentation clips so you don't need to hand-collect hours of audio. This pulls the standard training bundle.

In [ ]:
# The upstream negative-feature bank is float16 with shape (5_625_000, 16, 96)
# -- ~16 GB. np.load()-ing it whole OOMs even a Colab GPU box (and a 16 GB
# download is wasteful), so fetch only a subset of the already-windowed
# negatives via HTTP Range. Each window is 16*96*2 = 3 KB.
import numpy as np, requests, os, ast
NEG_URL = ("https://huggingface.co/datasets/davidscripka/openwakeword_features/"
           "resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy")
N_NEG = 200_000          # windows to use; raise for a stronger model (~3 KB each)
os.makedirs('out', exist_ok=True)

# Parse the .npy header to learn dtype/shape and where the raw data starts.
head = requests.get(NEG_URL, headers={'Range': 'bytes=0-127'}, timeout=60).content
assert head[:6] == b'\x93NUMPY', 'unexpected negatives format'
hlen = int.from_bytes(head[8:10], 'little'); data_off = 10 + hlen
meta = ast.literal_eval(head[10:10 + hlen].decode('latin1'))  # descr/fortran_order/shape
total, win, dim = meta['shape']; itemsize = np.dtype(meta['descr']).itemsize
n = min(N_NEG, total); end = data_off + n * win * dim * itemsize - 1
buf = requests.get(NEG_URL, headers={'Range': f'bytes={data_off}-{end}'},
                   timeout=1200).content
neg = np.frombuffer(buf, dtype=meta['descr']).reshape(n, win, dim)
np.save('out/negative_features.npy', neg)
print('negatives staged:', neg.shape, neg.dtype)

## 5. Compute features for the positive clips

In [ ]:
import numpy as np, glob, scipy.io.wavfile as wav
from scipy.signal import resample_poly
from openwakeword.utils import AudioFeatures
fe = AudioFeatures(inference_framework='onnx')  # tflite-runtime isn't installed

TARGET_SR = 16000
MIN_SAMPLES = int(2.5 * TARGET_SR)   # pad short clips so we get >= 16 embedding frames

def load_16k(path):
    sr, data = wav.read(path)
    if data.ndim > 1:                       # stereo -> mono
        data = data.mean(axis=1)
    if data.dtype.kind == 'f':              # float [-1,1] -> int16
        data = (np.clip(data, -1, 1) * 32767).astype(np.int16)
    else:
        data = data.astype(np.int16)
    if sr != TARGET_SR:                     # openWakeWord features assume 16 kHz
        data = resample_poly(data.astype(np.float32), TARGET_SR, sr).astype(np.int16)
    if len(data) < MIN_SAMPLES:             # centre-pad with silence
        pad = MIN_SAMPLES - len(data)
        data = np.pad(data, (pad // 2, pad - pad // 2))
    return data

def clip_to_features(path):
    # embed_clips wants a 2-D ndarray (n_clips, n_samples), not a list
    return fe.embed_clips(np.array([load_16k(path)]), batch_size=1)[0]

# pos is a list of (T, 96) arrays (T varies per clip); negatives are pre-windowed.
pos = [clip_to_features(p) for p in glob.glob('out/positives/*.wav')]
neg = np.load('out/negative_features.npy')
print('positive clips:', len(pos),
      '| frames/clip e.g.:', (pos[0].shape if pos else None),
      '| negatives:', neg.shape)

## 6. Train the classifier

In [ ]:
import numpy as np, tensorflow as tf
from tensorflow.keras import layers, models

# openWakeWord classifies a sliding 16 x 96 window of embeddings.
WIN = 16
def windows_from_seqs(seqs):
    out = []
    for seq in seqs:
        seq = np.asarray(seq, dtype=np.float32)
        if len(seq) < WIN:                       # left-pad a short clip to one window
            out.append(np.pad(seq, ((WIN - len(seq), 0), (0, 0))))
        else:
            for i in range(len(seq) - WIN + 1):  # slide
                out.append(seq[i:i + WIN])
    return np.asarray([w for w in out if w.shape == (WIN, 96)], dtype=np.float32)

Xp = windows_from_seqs(pos)                       # positives -> sliding windows
Xn = np.asarray(neg, dtype=np.float32)            # negatives are already (N, 16, 96)
print('positive windows:', Xp.shape, '| negative windows:', Xn.shape)
assert len(Xp) and len(Xn), 'empty class -- check positive/negative generation above'

X = np.concatenate([Xp, Xn])
y = np.concatenate([np.ones(len(Xp)), np.zeros(len(Xn))])

model = models.Sequential([
    layers.Input((WIN, 96)), layers.Flatten(),
    layers.Dense(128, activation='relu'), layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid'),
])
model.compile('adam', 'binary_crossentropy', metrics=['accuracy'])
model.fit(X, y, epochs=30, batch_size=256, validation_split=0.1,
          class_weight={0: 1.0, 1: len(Xn) / max(1, len(Xp))})

## 7. Export to ONNX

In [ ]:
!pip install -q tf2onnx
import tf2onnx, tensorflow as tf
spec = (tf.TensorSpec((None, WIN, 96), tf.float32, name='input'),)
tf2onnx.convert.from_keras(model, input_signature=spec, opset=13,
                           output_path='hey_friday.onnx')
print('exported hey_friday.onnx')

## 8. Quick sanity check + download

In [ ]:
import onnxruntime as ort, numpy as np
sess = ort.InferenceSession('hey_friday.onnx')
score = sess.run(None, {'input': Xp[:1].astype('float32')})[0]
print('score on a positive window (want ~1.0):', float(score[0][0]))
try:
    from google.colab import files   # download when running on Colab
    files.download('hey_friday.onnx')
except Exception:
    print('Saved hey_friday.onnx in the working directory.')

## 9. Use it in FRIDAY

```bash
mkdir -p models/wakeword && mv ~/Downloads/hey_friday.onnx models/wakeword/
# .env
FRIDAY_ENABLE_WAKEWORD=true
FRIDAY_WAKEWORD_MODEL=models/wakeword/hey_friday.onnx
```

The server wake-word seam (`friday.voice.wake_engine`) lazy-loads this ONNX via openWakeWord, and on a "hey friday" detection pushes a wake event to the HUD over the existing voice WebSocket — which reveals the cockpit and has FRIDAY greet you in her voice.